In [1]:
# Spec A: training-median imputation and scaling, without missingness indicators.
# Spec B: the same processing plus informative original/market missingness indicators.
# Missing market data stays missing until median imputation; it is never filled with zero.
# Fit each specification independently within every training fold.

In [2]:
# Locate shared project configuration and reusable helpers.
import sys as _sys
from pathlib import Path as _Path

_UTILS_ROOT = next(
    path
    for path in (_Path.cwd().resolve(), *_Path.cwd().resolve().parents)
    if (path / "notebook_utils.py").is_file()
)
if str(_UTILS_ROOT) not in _sys.path:
    _sys.path.insert(0, str(_UTILS_ROOT))
from notebook_utils import project_paths, file_hash
from project_config import ID_COLUMNS
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, MissingIndicator
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

ROOT = Path.cwd().resolve()
if not (ROOT / "data/processed/modeling_dataset_unprocessed.csv").exists():
    ROOT = ROOT.parent
PROCESSED = project_paths(ROOT).processed
MODELS = project_paths(ROOT).models
KEYS = ["company_id", "prediction_date"]
SPLITS = ["train", "validation", "test"]
NUMERIC_PREFIXES = ("original_", "derived_", "trend_", "market_", "macro_")
CATEGORICAL_COLUMNS = ["sector", "industry"]
MISSING_CATEGORY = "__MISSING__"
input_hashes = {}


def read_input(path):
    input_hashes[path] = file_hash(path)
    return pd.read_csv(
        path, dtype={"company_id": "string", "metadata_event_id": "string"}
    )


model = read_input(PROCESSED / "modeling_dataset_unprocessed.csv")
if (
    model.empty
    or model[KEYS].isna().any().any()
    or model.duplicated(KEYS).any()
    or not model.metadata_event_id.is_unique
):
    raise ValueError("Require uniquely identified company-events")
model_before = model.copy(deep=True)
ids = {name: read_input(PROCESSED / f"{name}_ids.csv") for name in SPLITS}
model_indexed = model.set_index(KEYS)
cohorts = {}
for name in SPLITS:
    frame = ids[name]
    if (
        not set(ID_COLUMNS).issubset(frame.columns)
        or frame.empty
        or frame[ID_COLUMNS].isna().any().any()
    ):
        raise ValueError(f"{name}: missing split identifiers/metadata")
    if (
        frame.duplicated(KEYS).any()
        or not frame.metadata_event_id.is_unique
        or not frame.split.eq(name).all()
    ):
        raise ValueError(f"{name}: invalid unique membership")
    index = pd.MultiIndex.from_frame(frame[KEYS])
    if len(index.difference(model_indexed.index)):
        raise ValueError(
            f"{name}: split contains a company-event absent from the dataset"
        )
    # Keep ID-file order without inserting columns into a fragmented wide DataFrame.
    values = model_indexed.reindex(index).copy()
    values.index = frame.index
    cohort = pd.concat([frame[KEYS], values], axis=1)
    if not cohort.target.isin([0, 1]).all():
        raise ValueError(f"{name}: nonbinary target")
    cohorts[name] = cohort
for i, left in enumerate(SPLITS):
    for right in SPLITS[i + 1 :]:
        for column in ["metadata_event_id", "company_id", "company_group_id"]:
            if not set(ids[left][column]).isdisjoint(ids[right][column]):
                raise ValueError(f"{column} leaks between {left} and {right}")
for left, right in [("train", "validation"), ("validation", "test")]:
    if (
        not pd.to_datetime(ids[left].prediction_date).max()
        < pd.to_datetime(ids[right].prediction_date).min()
    ):
        raise ValueError("Prediction chronology violated")
    if (
        not pd.to_datetime(ids[left].metadata_outcome_end_date).max()
        < pd.to_datetime(ids[right].prediction_date).min()
    ):
        raise ValueError(
            "Earlier split's outcome labels are not available before the next split"
        )

numeric_candidates = [column for column in model if column.startswith(NUMERIC_PREFIXES)]
if not numeric_candidates or not set(CATEGORICAL_COLUMNS).issubset(model.columns):
    raise ValueError("Missing expected numerical/categorical predictors")
if not all(
    pd.api.types.is_numeric_dtype(model[column]) for column in numeric_candidates
):
    raise ValueError("Numerical predictors must already have numerical types")
for name, cohort in cohorts.items():
    if np.isinf(cohort[numeric_candidates].to_numpy(dtype=float)).any():
        raise ValueError(
            f"{name}: infinite source predictors require upstream correction"
        )
    for column in [c for c in cohort if c.startswith("missing_")]:
        original_column = "original_" + column.removeprefix("missing_")
        if (
            original_column not in cohort
            or not cohort[column].eq(cohort[original_column].isna().astype(int)).all()
        ):
            raise ValueError(f"{name}: inconsistent original missingness flag {column}")

# This is the only data used to select columns/vocabularies or fit parameters.
X = {
    name: cohort[numeric_candidates + CATEGORICAL_COLUMNS].copy()
    for name, cohort in cohorts.items()
}
training_observed = X["train"][numeric_candidates].notna().any()
numeric_columns = training_observed.index[training_observed].tolist()
dropped_numeric_columns = training_observed.index[~training_observed].tolist()
if not numeric_columns:
    raise ValueError("No observed training numerical predictors")
category_vocabularies = {}
for column in CATEGORICAL_COLUMNS:
    observed = X["train"][column].dropna()
    if not observed.map(lambda value: isinstance(value, str)).all():
        raise ValueError(f"{column}: expected string categories")
    if observed.eq(MISSING_CATEGORY).any():
        raise ValueError(f"Reserved category collides with observed {column}")
    category_vocabularies[column] = sorted(observed.unique().tolist()) + [
        MISSING_CATEGORY
    ]

In [3]:
# Specification A has no indicators. Specification B adds indicators only for informative
# original financial and market fields, selected from the current training rows.
INDICATOR_PREFIXES = ("original_", "market_")


def informative_indicator_columns(training_frame, columns):
    candidates = [c for c in columns if c.startswith(INDICATOR_PREFIXES)]
    return [
        c
        for c in candidates
        if training_frame[c].isna().any() and training_frame[c].notna().any()
    ]


def build_preprocessing_pipeline(
    training_frame, numeric_fields, vocabularies, specification
):
    indicator_fields = (
        informative_indicator_columns(training_frame, numeric_fields)
        if specification == "B"
        else []
    )
    numeric_transform = Pipeline(
        [
            ("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("standardize", StandardScaler()),
        ]
    )
    categorical_transform = Pipeline(
        [
            (
                "impute",
                SimpleImputer(
                    strategy="constant",
                    fill_value=MISSING_CATEGORY,
                    keep_empty_features=True,
                ),
            ),
            (
                "one_hot",
                OneHotEncoder(
                    categories=[vocabularies[c] for c in CATEGORICAL_COLUMNS],
                    handle_unknown="ignore",
                    sparse_output=False,
                    dtype=np.float64,
                ),
            ),
        ]
    )
    branches = [("numeric", numeric_transform, numeric_fields)]
    if indicator_fields:
        branches.append(
            (
                "missing",
                MissingIndicator(features="all", sparse=False, error_on_new=False),
                indicator_fields,
            )
        )
    branches.append(("categorical", categorical_transform, CATEGORICAL_COLUMNS))
    transformer = ColumnTransformer(
        branches, remainder="drop", sparse_threshold=0, verbose_feature_names_out=True
    )
    pipeline = Pipeline([("preprocess", transformer)])
    pipeline.fit(training_frame[numeric_fields + CATEGORICAL_COLUMNS])
    names = pipeline.get_feature_names_out().tolist()
    if len(names) != len(set(names)):
        raise ValueError(
            f"Specification {specification}: duplicate transformed feature names"
        )
    pipeline.project_schema_ = {
        "specification": specification,
        "input_feature_columns": numeric_candidates + CATEGORICAL_COLUMNS,
        "retained_numeric_columns": numeric_fields,
        "dropped_training_empty_numeric_columns": dropped_numeric_columns
        if training_frame is X["train"]
        else [],
        "categorical_columns": CATEGORICAL_COLUMNS,
        "category_vocabularies": vocabularies,
        "missing_category": MISSING_CATEGORY,
        "indicator_columns": indicator_fields,
        "matrix_feature_columns": names,
        "matrix_metadata_columns": ID_COLUMNS,
        "fit_split": "train",
        "training_event_ids": ids["train"].metadata_event_id.tolist(),
        "source_sha256": input_hashes[PROCESSED / "modeling_dataset_unprocessed.csv"],
        "split_file_sha256": {
            name: input_hashes[PROCESSED / f"{name}_ids.csv"] for name in SPLITS
        },
        "split_counts": {
            name: {"events": len(ids[name]), "bankrupt": int(ids[name].target.sum())}
            for name in SPLITS
        },
        "unseen_category_policy": "all-zero one-hot block (not missing)",
        "missingness_policy": "none"
        if specification == "A"
        else "training-varying original/market fields only",
        "numeric_imputation": "training-fold median; unavailable values remain missing until pipeline transform",
        "numeric_scaling": "standardized using training-fold statistics",
        "matrix_role": "Predictors plus separate ID/target metadata; no classifier fitted",
    }
    return pipeline, indicator_fields


preprocessing_pipelines = {}
preprocessing_feature_names = {}
preprocessing_indicators = {}
transformed_by_specification = {}
for specification in ["A", "B"]:
    pipeline, indicator_fields = build_preprocessing_pipeline(
        X["train"], numeric_columns, category_vocabularies, specification
    )
    preprocessing_pipelines[specification] = pipeline
    preprocessing_feature_names[specification] = (
        pipeline.get_feature_names_out().tolist()
    )
    preprocessing_indicators[specification] = indicator_fields
    transformed_by_specification[specification] = {
        name: pipeline.transform(X[name]) for name in SPLITS
    }

# Keep Spec B on the established paths used by downstream notebooks.
preprocessing_pipeline = preprocessing_pipelines["B"]
feature_names = preprocessing_feature_names["B"]
transformed = transformed_by_specification["B"]
state_after_fit = joblib.hash(preprocessing_pipeline)

In [4]:
# Verify each fitted parameter directly against the training cohort.
fitted_columns = preprocessing_pipeline.named_steps["preprocess"]
fitted_imputer = fitted_columns.named_transformers_["numeric"].named_steps["impute"]
fitted_scaler = fitted_columns.named_transformers_["numeric"].named_steps["standardize"]
fitted_indicator = fitted_columns.named_transformers_["missing"]
fitted_encoder = fitted_columns.named_transformers_["categorical"].named_steps[
    "one_hot"
]
expected_medians = X["train"][numeric_columns].median(axis=0).to_numpy(dtype=float)
expected_imputed = (
    X["train"][numeric_columns]
    .fillna(pd.Series(expected_medians, index=numeric_columns))
    .to_numpy(dtype=float)
)

numeric_slice = fitted_columns.output_indices_["numeric"]
indicator_slice = fitted_columns.output_indices_["missing"]
categorical_slice = fitted_columns.output_indices_["categorical"]
for name in SPLITS:
    values = np.asarray(transformed[name], dtype=float)
    indicators = values[:, indicator_slice]
    expected_scaled = (
        X[name][numeric_columns]
        .fillna(pd.Series(expected_medians, index=numeric_columns))
        .to_numpy(dtype=float)
        - fitted_scaler.mean_
    ) / fitted_scaler.scale_
constant = fitted_scaler.var_ == 0

# Preserve the training-fitted state after transforming held-out rows.

# Report unknown held-out categories without modifying the fitted vocabulary.
category_coverage = []
for name in SPLITS:
    for column in CATEGORICAL_COLUMNS:
        values = X[name][column]
        unknown = values.notna() & ~values.isin(category_vocabularies[column])
        category_coverage.append(
            {
                "split": name,
                "categorical_field": column,
                "missing_rows": int(values.isna().sum()),
                "unseen_category_rows": int(unknown.sum()),
                "unseen_values": sorted(values.loc[unknown].unique().tolist()),
            }
        )

In [5]:
# Save both named specifications. Existing matrix and pipeline paths remain Spec B.
MODELS.mkdir(parents=True, exist_ok=True)
for specification in ["A", "B"]:
    pipeline = preprocessing_pipelines[specification]
    spec_model_path = (
        MODELS / f"preprocessing_pipeline_spec_{specification.lower()}.joblib"
    )
    joblib.dump(pipeline, spec_model_path, compress=3)
    spec_output = PROCESSED / f"preprocessing_spec_{specification.lower()}"
    spec_output.mkdir(parents=True, exist_ok=True)
    spec_features = preprocessing_feature_names[specification]
    for name in SPLITS:
        predictor_matrix = pd.DataFrame(
            transformed_by_specification[specification][name],
            columns=spec_features,
            index=ids[name].index,
        )
        matrix = pd.concat([ids[name][ID_COLUMNS].copy(), predictor_matrix], axis=1)
        matrix.to_csv(spec_output / f"{name}_matrix.csv", index=False)
        if specification == "B":
            matrix.to_csv(PROCESSED / f"{name}_matrix.csv", index=False)

artifact_path = MODELS / "preprocessing_pipeline.joblib"
joblib.dump(preprocessing_pipelines["B"], artifact_path, compress=3)
loaded_pipeline = joblib.load(artifact_path)

In [6]:
# Fit both specifications independently inside each walk-forward training fold.
walk_forward_ids_path = PROCESSED / "walk_forward_fold_ids.csv"
input_hashes[walk_forward_ids_path] = file_hash(walk_forward_ids_path)
walk_forward_membership = pd.read_csv(
    walk_forward_ids_path, dtype={"company_id": "string", "metadata_event_id": "string"}
)
walk_forward_membership["metadata_outcome_end_date"] = pd.to_datetime(
    walk_forward_membership.metadata_outcome_end_date, errors="raise"
).astype("datetime64[ns]")
walk_forward_root = PROCESSED / "walk_forward"
walk_forward_root.mkdir(parents=True, exist_ok=True)
walk_forward_model_root = MODELS / "walk_forward"
walk_forward_model_root.mkdir(parents=True, exist_ok=True)
walk_forward_summaries = []

for fold_name in sorted(
    walk_forward_membership.loc[
        walk_forward_membership.fold.ne("final_test"), "fold"
    ].unique()
):
    fold_members = walk_forward_membership.loc[
        walk_forward_membership.fold.eq(fold_name)
    ]
    fold_train_ids = fold_members.loc[fold_members.split.eq("train")].copy()
    fold_validation_ids = fold_members.loc[fold_members.split.eq("validation")].copy()
    if fold_train_ids.empty or fold_validation_ids.empty:
        raise ValueError(f"{fold_name}: missing train or validation events")
    if set(fold_train_ids.company_group_id).intersection(
        fold_validation_ids.company_group_id
    ):
        raise ValueError(f"{fold_name}: company group crosses the fold boundary")
    if (
        not pd.to_datetime(fold_train_ids.metadata_outcome_end_date).max()
        < pd.to_datetime(fold_validation_ids.prediction_date).min()
    ):
        raise ValueError(f"{fold_name}: training outcome window reaches validation")

    fold_X = {}
    for split_name, ids_frame in [
        ("train", fold_train_ids),
        ("validation", fold_validation_ids),
    ]:
        fold_index = pd.MultiIndex.from_frame(ids_frame[KEYS])
        absent = fold_index.difference(model_indexed.index)
        if len(absent):
            raise ValueError(
                f"{fold_name}/{split_name}: events absent from modeling data"
            )
        aligned = model_indexed.reindex(fold_index)
        fold_X[split_name] = aligned[numeric_candidates + CATEGORICAL_COLUMNS].copy()
        if not ids_frame.target.isin([0, 1]).all():
            raise ValueError(f"{fold_name}/{split_name}: invalid labels")

    observed = fold_X["train"][numeric_candidates].notna().any()
    fold_numeric_columns = observed.index[observed].tolist()
    fold_dropped_numeric = observed.index[~observed].tolist()
    if not fold_numeric_columns:
        raise ValueError(f"{fold_name}: no observed training numerical features")
    fold_vocabularies = {}
    for column in CATEGORICAL_COLUMNS:
        categories = fold_X["train"][column].dropna()
        if not categories.map(lambda value: isinstance(value, str)).all():
            raise ValueError(f"{fold_name}/{column}: expected string categories")
        if categories.eq(MISSING_CATEGORY).any():
            raise ValueError(
                f"{fold_name}: reserved missing category collides with {column}"
            )
        fold_vocabularies[column] = sorted(categories.unique().tolist()) + [
            MISSING_CATEGORY
        ]

    for specification in ["A", "B"]:
        indicator_fields = (
            informative_indicator_columns(fold_X["train"], fold_numeric_columns)
            if specification == "B"
            else []
        )
        numeric_transform = Pipeline(
            [
                ("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
                ("standardize", StandardScaler()),
            ]
        )
        categorical_transform = Pipeline(
            [
                (
                    "impute",
                    SimpleImputer(
                        strategy="constant",
                        fill_value=MISSING_CATEGORY,
                        keep_empty_features=True,
                    ),
                ),
                (
                    "one_hot",
                    OneHotEncoder(
                        categories=[fold_vocabularies[c] for c in CATEGORICAL_COLUMNS],
                        handle_unknown="ignore",
                        sparse_output=False,
                        dtype=np.float64,
                    ),
                ),
            ]
        )
        branches = [("numeric", numeric_transform, fold_numeric_columns)]
        if indicator_fields:
            branches.append(
                (
                    "missing",
                    MissingIndicator(features="all", sparse=False, error_on_new=False),
                    indicator_fields,
                )
            )
        branches.append(("categorical", categorical_transform, CATEGORICAL_COLUMNS))
        transformer = ColumnTransformer(
            branches,
            remainder="drop",
            sparse_threshold=0,
            verbose_feature_names_out=True,
        )
        fold_pipeline = Pipeline([("preprocess", transformer)])
        train_frame = fold_X["train"][fold_numeric_columns + CATEGORICAL_COLUMNS]
        fold_pipeline.fit(train_frame)
        fold_feature_names = fold_pipeline.get_feature_names_out().tolist()
        if len(fold_feature_names) != len(set(fold_feature_names)):
            raise ValueError(
                f"{fold_name}/{specification}: duplicate transformed feature names"
            )
        fold_pipeline.project_schema_ = {
            "specification": specification,
            "input_feature_columns": numeric_candidates + CATEGORICAL_COLUMNS,
            "retained_numeric_columns": fold_numeric_columns,
            "dropped_training_empty_numeric_columns": fold_dropped_numeric,
            "categorical_columns": CATEGORICAL_COLUMNS,
            "category_vocabularies": fold_vocabularies,
            "missing_category": MISSING_CATEGORY,
            "indicator_columns": indicator_fields,
            "matrix_feature_columns": fold_feature_names,
            "matrix_metadata_columns": ID_COLUMNS,
            "fit_split": f"{fold_name}/train",
            "training_event_ids": fold_train_ids.metadata_event_id.tolist(),
            "source_sha256": input_hashes[
                PROCESSED / "modeling_dataset_unprocessed.csv"
            ],
            "fold_ids_sha256": input_hashes[walk_forward_ids_path],
            "unseen_category_policy": "all-zero one-hot block (not missing)",
            "missingness_policy": "none"
            if specification == "A"
            else "training-varying original/market fields only",
            "numeric_imputation": "training-fold median",
            "numeric_scaling": "training-fold standardization",
        }

        spec_name = f"spec_{specification.lower()}"
        fold_output = walk_forward_root / fold_name / spec_name
        fold_output.mkdir(parents=True, exist_ok=True)
        fold_model_path = (
            walk_forward_model_root
            / f"{fold_name}_{spec_name}_preprocessing_pipeline.joblib"
        )
        joblib.dump(fold_pipeline, fold_model_path, compress=3)
        for split_name, ids_frame in [
            ("train", fold_train_ids),
            ("validation", fold_validation_ids),
        ]:
            fold_index = pd.MultiIndex.from_frame(ids_frame[KEYS])
            values = model_indexed.reindex(fold_index)
            predictors = fold_pipeline.transform(
                values[numeric_candidates + CATEGORICAL_COLUMNS]
            )
            matrix = pd.DataFrame(predictors, columns=fold_feature_names)
            matrix = pd.concat(
                [ids_frame[ID_COLUMNS].reset_index(drop=True), matrix], axis=1
            )
            matrix.to_csv(fold_output / f"{split_name}_matrix.csv", index=False)
            if specification == "B":
                legacy_output = walk_forward_root / fold_name
                legacy_output.mkdir(parents=True, exist_ok=True)
                matrix.to_csv(legacy_output / f"{split_name}_matrix.csv", index=False)
        walk_forward_summaries.append(
            {
                "fold": fold_name,
                "specification": specification,
                "train_rows": len(fold_train_ids),
                "train_bankruptcies": int(fold_train_ids.target.sum()),
                "validation_rows": len(fold_validation_ids),
                "validation_bankruptcies": int(fold_validation_ids.target.sum()),
                "retained_numeric_features": len(fold_numeric_columns),
                "indicator_features": len(indicator_fields),
                "transformed_features": len(fold_feature_names),
                "pipeline_path": str(fold_model_path.relative_to(ROOT)),
            }
        )

walk_forward_preprocessing_summary = pd.DataFrame(walk_forward_summaries)
walk_forward_preprocessing_summary.to_csv(
    walk_forward_root / "preprocessing_summary.csv", index=False
)